# Prototype Algorithm: Rules-Based Customer Segmentation

This notebook implements a **rules-based algorithm** that assigns customers to segments using **K-Prototypes clustering (4 clusters)** and provides segment-specific messages, discounts, and product recommendations.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder
from kmodes.kprototypes import KPrototypes
import warnings
warnings.filterwarnings('ignore')

# Plot settings
sns.set(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)

# Data directory - Modified to your specific path
data_dir = '/Users/t0m/Downloads/Ecommerce data'

# Load training data (sum_18_final.csv) and test data (2020-21)
sum_18_final_path = os.path.join(data_dir, "sum_18_final.csv")
summary_2021_path = os.path.join(data_dir, "summary2020-21.csv")


# if not os.path.exists(sum_18_final_path):
#     sum_18_final_path = os.path.join(os.path.dirname(data_dir), "sum_18_final.csv")

# Read the files
try:
    sum_18_final = pd.read_csv(sum_18_final_path)
    summary_2021 = pd.read_csv(summary_2021_path)

    print("Data loaded successfully:")
    print(f"  sum_18_final (training): {sum_18_final.shape}")
    print(f"  summary_2021 (test): {summary_2021.shape}")
    print(f"\nTraining data columns: {sum_18_final.columns.tolist()[:10]}...")

except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    print("请确认 'sum_18_final.csv' 和 'summary2020-21.csv' 是否都在 '/Users/t0m/Downloads/Ecommerce data' 目录下。")

## Step 1: Prepare Training Data (sum_18_final.csv)

Using the same features as in `testing.ipynb`:
- **Numeric**: avgdailyspend, totalspend, purchasedays, totalpurchases, avg_quantity, category_diversity, avg_purchase_price, top_category_concentration
- **Categorical**: Q-amazon-use-howmany, Q-amazon-use-how-oft, Q-demos-age, Q-demos-income


In [ ]:
# --- Prepare training data (sum_18_final.csv) ---

# Use the same features as in testing.ipynb
df_train = sum_18_final.copy()

# Drop columns with too many missing values and non-feature columns
df_train = df_train.dropna(axis=1).drop(columns=['respondentid', 'firstpurchase', 'lastpurchase'], errors='ignore')

# Define features for K-Prototypes (same as testing.ipynb)
numeric_cols = [
    'avgdailyspend', 'totalspend', 'purchasedays', 'totalpurchases',
    'avg_quantity', 'category_diversity', 'avg_purchase_price',
    'top_category_concentration'
]

categorical_cols = [
    'Q-amazon-use-howmany', 'Q-amazon-use-how-oft', 'Q-demos-age', 'Q-demos-income'
]

# Filter to only selected features that exist
available_numeric = [col for col in numeric_cols if col in df_train.columns]
available_categorical = [col for col in categorical_cols if col in df_train.columns]

print(f"Available numeric features: {available_numeric}")
print(f"Available categorical features: {available_categorical}")

# Prepare data for K-Prototypes
df_prep = df_train[available_numeric + available_categorical].copy()
df_prep = df_prep.dropna()

print(f"\nPrepared training data shape: {df_prep.shape}")
print(f"Numeric features: {len(available_numeric)}, Categorical features: {len(available_categorical)}")


## Step 2: Fit K-Prototypes (k=4) on Training Data


In [ ]:
# --- Prepare data for K-Prototypes ---

# Scale numeric columns
scaler = StandardScaler()
df_prep_scaled = df_prep.copy()
df_prep_scaled[available_numeric] = scaler.fit_transform(df_prep[available_numeric])

# Encode categorical columns
label_encoders = {}
for col in available_categorical:
    le = LabelEncoder()
    df_prep_scaled[col] = le.fit_transform(df_prep_scaled[col].astype(str))
    label_encoders[col] = le

# Convert to numpy array
data_array = df_prep_scaled.to_numpy()

# Get categorical column indices (for K-Prototypes)
cat_idx = [df_prep_scaled.columns.get_loc(col) for col in available_categorical]

print(f"Data array shape: {data_array.shape}")
print(f"Categorical column indices: {cat_idx}")


## Step 3: Run K-Prototypes Clustering


In [ ]:
# --- Fit K-Prototypes with k=4 on training data ---

kproto = KPrototypes(n_clusters=4, init='Cao', random_state=42, n_init=10)
clusters = kproto.fit_predict(data_array, categorical=cat_idx)

# Add cluster assignments back to original (unscaled) data
df_train_clean = df_train.loc[df_prep.index].copy()
df_train_clean["cluster"] = clusters

print("K-Prototypes clustering complete (k=4) on training data")
print(f"\nCluster distribution:")
print(df_train_clean["cluster"].value_counts().sort_index())

# Cluster profiles
print("\n" + "="*80)
print("CLUSTER PROFILES (Training Data)")
print("="*80)
cluster_profiles = df_train_clean.groupby("cluster")[available_numeric].agg(['mean', 'median']).round(2)
print(cluster_profiles)


## Step 4: Classify Clusters into 4 Segment Types

We'll compare clusters directly to each other to assign them to the 4 segment types:
1. **High-Value, High-Engagement**: Highest spend + highest frequency + lowest recency
2. **Low-Value, Low-Engagement**: Lowest spend + lowest frequency + highest recency
3. **High-Frequency, Low-Spend**: Highest frequency but lowest spend per purchase
4. **Mid-Tier Stable Buyers**: The remaining cluster (moderate across dimensions)


In [ ]:
# --- Classify clusters by comparing them directly to each other ---

# Calculate key metrics for classification
# We need: totalspend, totalpurchases, recency (need to calculate), spend_per_purchase
df_train_clean["spend_per_purchase"] = df_train_clean["totalspend"] / df_train_clean["totalpurchases"].replace(0, 1)

# For recency, calculate from dates if available
if "lastpurchase" in sum_18_final.columns:
    df_train_clean["lastpurchase"] = pd.to_datetime(sum_18_final.loc[df_prep.index, "lastpurchase"])
    ref_date = datetime(2019, 12, 31)
    df_train_clean["recency_days"] = (ref_date - df_train_clean["lastpurchase"]).dt.days
else:
    # Use purchasedays as proxy - more purchasedays relative to totalpurchases = more active = lower recency
    # For simplicity, we'll use a normalized measure: recency_days = purchasedays (inverted logic)
    df_train_clean["recency_days"] = df_train_clean["purchasedays"]

# Get cluster centroids with all key metrics
cluster_metrics = df_train_clean.groupby("cluster").agg({
    "totalspend": "mean",
    "totalpurchases": "mean",
    "recency_days": "mean",
    "spend_per_purchase": "mean"
}).round(2)

print("Cluster Metrics (for classification):")
print(cluster_metrics)
print()

# Classify each cluster by comparing to others
segment_strategies = {}

# 1. Find High-Value, High-Engagement: highest spend AND highest frequency AND lowest recency
# Calculate composite score: (spend + freq) - recency (higher is better)
cluster_metrics["composite_score"] = (
    cluster_metrics["totalspend"] / cluster_metrics["totalspend"].max() +
    cluster_metrics["totalpurchases"] / cluster_metrics["totalpurchases"].max() -
    cluster_metrics["recency_days"] / cluster_metrics["recency_days"].max()
)
high_value_cluster = cluster_metrics["composite_score"].idxmax()

# 2. Find Low-Value, Low-Engagement: lowest spend AND lowest frequency AND highest recency
# Calculate composite score: -(spend + freq) + recency (higher is worse)
cluster_metrics["low_value_score"] = (
    -cluster_metrics["totalspend"] / cluster_metrics["totalspend"].max() -
    cluster_metrics["totalpurchases"] / cluster_metrics["totalpurchases"].max() +
    cluster_metrics["recency_days"] / cluster_metrics["recency_days"].max()
)
low_value_cluster = cluster_metrics["low_value_score"].idxmax()

# 3. Find High-Frequency, Low-Spend: highest frequency but lowest spend_per_purchase
# Calculate score: frequency / spend_per_purchase (higher is better)
cluster_metrics["freq_low_spend_score"] = (
    cluster_metrics["totalpurchases"] / cluster_metrics["totalpurchases"].max() -
    cluster_metrics["spend_per_purchase"] / cluster_metrics["spend_per_purchase"].max()
)
high_freq_low_spend_cluster = cluster_metrics["freq_low_spend_score"].idxmax()

# 4. Mid-Tier Stable Buyers: the remaining cluster
remaining_clusters = set(cluster_metrics.index) - {high_value_cluster, low_value_cluster, high_freq_low_spend_cluster}
mid_tier_cluster = list(remaining_clusters)[0] if remaining_clusters else None

# If there's overlap, refine the assignment
if high_freq_low_spend_cluster == high_value_cluster or high_freq_low_spend_cluster == low_value_cluster:
    # Recalculate: prioritize frequency/spend_per_purchase ratio
    cluster_metrics["freq_spend_ratio"] = cluster_metrics["totalpurchases"] / (cluster_metrics["spend_per_purchase"] + 1)
    high_freq_low_spend_cluster = cluster_metrics["freq_spend_ratio"].idxmax()
    # Exclude already assigned clusters
    if high_freq_low_spend_cluster == high_value_cluster:
        temp = cluster_metrics["freq_spend_ratio"].copy()
        temp.loc[high_value_cluster] = -1
        high_freq_low_spend_cluster = temp.idxmax()
    if high_freq_low_spend_cluster == low_value_cluster:
        temp = cluster_metrics["freq_spend_ratio"].copy()
        temp.loc[low_value_cluster] = -1
        high_freq_low_spend_cluster = temp.idxmax()

# Recalculate mid-tier
remaining_clusters = set(cluster_metrics.index) - {high_value_cluster, low_value_cluster, high_freq_low_spend_cluster}
mid_tier_cluster = list(remaining_clusters)[0] if remaining_clusters else None

print("Cluster Classification:")
print(f"  High-Value, High-Engagement: Cluster {high_value_cluster}")
print(f"  Low-Value, Low-Engagement: Cluster {low_value_cluster}")
print(f"  High-Frequency, Low-Spend: Cluster {high_freq_low_spend_cluster}")
print(f"  Mid-Tier Stable Buyers: Cluster {mid_tier_cluster}")
print()

# Assign strategies to each cluster
for cluster_id in sorted(cluster_metrics.index):
    freq = cluster_metrics.loc[cluster_id, "totalpurchases"]
    spend = cluster_metrics.loc[cluster_id, "totalspend"]
    recency = cluster_metrics.loc[cluster_id, "recency_days"]
    spend_per_purchase = cluster_metrics.loc[cluster_id, "spend_per_purchase"]
    
    if cluster_id == high_value_cluster:
        segment_type = "High-Value, High-Engagement Customers"
        message = "Thank you for being a valued customer! Here's a personalized recommendation just for you."
        discount = 0
        recommendation_type = "Category-matched recommendations without discounts; loyalty programs"
    elif cluster_id == low_value_cluster:
        segment_type = "Low-Value, Low-Engagement Customers"
        message = "Welcome back! We have a special offer to get you started."
        discount = 20
        recommendation_type = "Strong price incentives, re-activation campaigns, discounted popular items"
    elif cluster_id == high_freq_low_spend_cluster:
        segment_type = "High-Frequency, Low-Spend Customers"
        message = "Love your frequent visits! Check out this bundle deal to maximize your savings."
        discount = 15
        recommendation_type = "Volume-based promotions, bundling, increase basket size"
    elif cluster_id == mid_tier_cluster:
        segment_type = "Mid-Tier Stable Buyers"
        message = "We appreciate your business! Here's a personalized suggestion just for you."
        discount = 10
        recommendation_type = "Light discounts, personalized suggestions, encourage upward movement"
    else:
        # Fallback (shouldn't happen with 4 clusters)
        segment_type = "Mid-Tier Stable Buyers"
        message = "We appreciate your business!"
        discount = 10
        recommendation_type = "Light discounts, personalized suggestions, encourage upward movement"
    
    segment_strategies[cluster_id] = {
        "segment_type": segment_type,
        "message": message,
        "discount": discount,
        "recommendation_type": recommendation_type,
        "avg_frequency": freq,
        "avg_spending": spend,
        "avg_recency": recency,
        "avg_spend_per_purchase": spend_per_purchase
    }
    
    print(f"Cluster {cluster_id}: {segment_type}")
    print(f"  Frequency: {freq:.1f}, Spending: ${spend:.2f}, Recency: {recency:.1f} days, Spend/Purchase: ${spend_per_purchase:.2f}")
    print(f"  Message: {message}")
    print(f"  Discount: {discount}%")
    print(f"  Recommendation: {recommendation_type}")
    print()

# Store for later use
SEGMENT_STRATEGIES = segment_strategies.copy()

# Calculate thresholds for rules-based assignment using cluster boundaries
# This is better than using medians - we use actual cluster min/max boundaries
print("\n" + "="*80)
print("CALCULATING THRESHOLDS FROM CLUSTER BOUNDARIES")
print("="*80)

threshold_features = ["totalspend", "totalpurchases", "recency_days", "spend_per_purchase"]
THRESHOLDS = {}

# For each feature, calculate cluster boundaries
for feature in threshold_features:
    if feature not in df_train_clean.columns:
        continue
    
    # Get min/max for each cluster
    cluster_stats = df_train_clean.groupby("cluster")[feature].agg(['min', 'max', 'mean']).round(2)
    
    # Sort clusters by mean value to identify low/medium/high tiers
    cluster_means = cluster_stats['mean'].sort_values()
    
    # Identify cluster tiers
    low_cluster = cluster_means.index[0]
    high_cluster = cluster_means.index[-1]
    medium_clusters = cluster_means.index[1:-1].tolist()
    
    # Calculate boundaries
    if len(medium_clusters) > 0:
        # Low threshold: boundary between low and medium clusters
        low_max = cluster_stats.loc[low_cluster, 'max']
        medium_min = cluster_stats.loc[medium_clusters, 'min'].min()
        threshold_low = (low_max + medium_min) / 2
        
        # High threshold: boundary between medium and high clusters
        medium_max = cluster_stats.loc[medium_clusters, 'max'].max()
        high_min = cluster_stats.loc[high_cluster, 'min']
        threshold_high = (medium_max + high_min) / 2
    else:
        # Only 2 clusters: use midpoint
        low_max = cluster_stats.loc[low_cluster, 'max']
        high_min = cluster_stats.loc[high_cluster, 'min']
        threshold_low = (low_max + high_min) / 2
        threshold_high = threshold_low
    
    # Store thresholds (we'll use low and high for classification)
    THRESHOLDS[f"{feature}_low"] = threshold_low
    THRESHOLDS[f"{feature}_high"] = threshold_high
    THRESHOLDS[f"{feature}_median"] = cluster_stats['mean'].median()  # Keep median for reference
    
    print(f"\n{feature}:")
    print(f"  Low cluster (ID {low_cluster}): min={cluster_stats.loc[low_cluster, 'min']:.2f}, max={cluster_stats.loc[low_cluster, 'max']:.2f}, mean={cluster_stats.loc[low_cluster, 'mean']:.2f}")
    if len(medium_clusters) > 0:
        print(f"  Medium clusters (IDs {medium_clusters}): min={cluster_stats.loc[medium_clusters, 'min'].min():.2f}, max={cluster_stats.loc[medium_clusters, 'max'].max():.2f}")
    print(f"  High cluster (ID {high_cluster}): min={cluster_stats.loc[high_cluster, 'min']:.2f}, max={cluster_stats.loc[high_cluster, 'max']:.2f}, mean={cluster_stats.loc[high_cluster, 'mean']:.2f}")
    print(f"  Threshold Low: {threshold_low:.2f}")
    print(f"  Threshold High: {threshold_high:.2f}")

print("\n" + "="*80)
print("THRESHOLDS FOR RULES-BASED ASSIGNMENT (from cluster boundaries)")
print("="*80)
for key, value in sorted(THRESHOLDS.items()):
    if not key.endswith('_median'):  # Only show low/high thresholds
        print(f"  {key}: {value:.2f}")


## Step 5: Rules-Based Assignment Function

This function assigns customers to segments in real-time using RFM-based rules.


In [ ]:
def assign_segment(customer_data, thresholds=None, segment_strategies=None):
    """
    Assign a customer to a segment based on RFM rules.
    
    Parameters:
    -----------
    customer_data : dict or pd.Series
        Customer features: totalspend, totalpurchases, recency_days, spend_per_purchase
    thresholds : dict (optional)
        Threshold values for each feature. If None, uses THRESHOLDS from above.
    segment_strategies : dict (optional)
        Segment strategies. If None, uses SEGMENT_STRATEGIES from above.
    
    Returns:
    --------
    dict with segment_id, segment_type, message, discount, recommendation_type
    """
    if thresholds is None:
        thresholds = THRESHOLDS
    if segment_strategies is None:
        segment_strategies = SEGMENT_STRATEGIES
    
    # Extract features
    freq = customer_data.get("totalpurchases", 0)
    spend = customer_data.get("totalspend", 0)
    recency = customer_data.get("recency_days", 999)
    spend_per_purchase = customer_data.get("spend_per_purchase", 0)
    if spend_per_purchase == 0 and freq > 0:
        spend_per_purchase = spend / freq
    
    # Get thresholds (using cluster boundaries - derived from actual cluster data, not arbitrary)
    freq_low = thresholds.get("totalpurchases_low", 0)
    freq_high = thresholds.get("totalpurchases_high", 1000)
    spend_low = thresholds.get("totalspend_low", 0)
    spend_high = thresholds.get("totalspend_high", 10000)
    recency_low = thresholds.get("recency_days_low", 50)
    recency_high = thresholds.get("recency_days_high", 200)
    spend_per_purchase_low = thresholds.get("spend_per_purchase_low", 20)
    spend_per_purchase_high = thresholds.get("spend_per_purchase_high", 50)
    
    # Classify based on cluster boundaries (more precise than arbitrary multipliers)
    is_high_freq = freq >= freq_high
    is_low_freq = freq <= freq_low
    is_high_spend = spend >= spend_high
    is_low_spend = spend <= spend_low
    is_low_recency = recency <= recency_low  # Low recency = recent purchase (good)
    is_high_recency = recency >= recency_high  # High recency = not recent (bad)
    is_low_spend_per_purchase = spend_per_purchase <= spend_per_purchase_low
    is_moderate_freq = freq_low < freq < freq_high
    is_moderate_spend = spend_low < spend < spend_high
    is_moderate_recency = recency_low < recency < recency_high
    
    # Determine segment type using cluster boundary logic (priority order)
    # 1. High-Value, High-Engagement: high spend AND high freq AND low recency
    if is_high_spend and is_high_freq and is_low_recency:
        segment_type = "High-Value, High-Engagement Customers"
    # 2. High-Frequency, Low-Spend: high freq but low spend per purchase
    elif is_high_freq and is_low_spend_per_purchase:
        segment_type = "High-Frequency, Low-Spend Customers"
    # 3. Low-Value, Low-Engagement: low spend AND low freq AND high recency
    elif is_low_spend and is_low_freq and is_high_recency:
        segment_type = "Low-Value, Low-Engagement Customers"
    # 4. Mid-Tier Stable Buyers: moderate across dimensions
    elif (is_moderate_freq or is_moderate_spend) and not (is_high_recency or is_low_recency):
        segment_type = "Mid-Tier Stable Buyers"
    # Fallback: use most prominent characteristic
    else:
        if is_high_freq and is_low_spend_per_purchase:
            segment_type = "High-Frequency, Low-Spend Customers"
        elif is_high_spend and is_high_freq:
            segment_type = "High-Value, High-Engagement Customers"
        elif is_low_spend and is_low_freq:
            segment_type = "Low-Value, Low-Engagement Customers"
        else:
            segment_type = "Mid-Tier Stable Buyers"
    
    # Find matching cluster strategy
    matching_cluster = None
    for cluster_id, strategy in segment_strategies.items():
        if strategy["segment_type"] == segment_type:
            matching_cluster = cluster_id
            break
    
    if matching_cluster is not None:
        strategy = segment_strategies[matching_cluster]
        return {
            "segment_id": matching_cluster,
            "segment_type": segment_type,
            "message": strategy["message"],
            "discount": strategy["discount"],
            "recommendation_type": strategy["recommendation_type"]
        }
    else:
        # Fallback to Low-Value, Low-Engagement
        return {
            "segment_id": 2,
            "segment_type": "Low-Value, Low-Engagement Customers",
            "message": "Welcome back! We have a special offer to get you started.",
            "discount": 20,
            "recommendation_type": "Strong price incentives, re-activation campaigns, discounted popular items"
        }

# Test the function
print("Testing assignment function:")
test_customer = {
    "totalspend": 5000,
    "totalpurchases": 200,
    "recency_days": 10,
    "spend_per_purchase": 25
}
result = assign_segment(test_customer)
print(f"\nTest customer assignment:")
for key, value in result.items():
    print(f"  {key}: {value}")


In [ ]:
def assign_new_customer():
    """
    Assign a new customer (no purchase history) to a segment.
    New customers are always assigned to Low-Value, Low-Engagement Customers segment initially.
    """
    # Find the cluster ID for Low-Value, Low-Engagement
    low_value_cluster_id = None
    for cluster_id, strategy in SEGMENT_STRATEGIES.items():
        if strategy["segment_type"] == "Low-Value, Low-Engagement Customers":
            low_value_cluster_id = cluster_id
            break
    
    if low_value_cluster_id is None:
        low_value_cluster_id = 2  # Fallback
    
    return {
        "segment_id": low_value_cluster_id,
        "segment_type": "Low-Value, Low-Engagement Customers",
        "message": "Welcome! As a new customer, enjoy 20% off your first purchase!",
        "discount": 20,
        "recommendation_type": "Strong price incentives, re-activation campaigns, discounted popular items",
        "is_new_customer": True
    }

# Test new customer assignment
print("New customer assignment:")
new_customer_result = assign_new_customer()
for key, value in new_customer_result.items():
    print(f"  {key}: {value}")


## Step 6: Update Function for Real-Time Re-Assignment

After each new transaction, customers can be re-assigned to segments based on updated metrics.


In [ ]:
def update_customer_segment(customer_id, new_transaction_data, current_customer_data=None):
    """
    Update customer segment after a new transaction.
    
    Parameters:
    -----------
    customer_id : str
        Customer identifier
    new_transaction_data : dict
        New transaction: {amount, quantity, category, date}
    current_customer_data : dict (optional)
        Current customer metrics: {totalspend, totalpurchases, purchasedays, last_purchase_date}
    
    Returns:
    --------
    dict with updated segment assignment and recommendations
    """
    # If this is a new customer, use new customer assignment
    if current_customer_data is None or current_customer_data.get("totalpurchases", 0) == 0:
        return assign_new_customer()
    
    # Update customer metrics
    updated_data = current_customer_data.copy()
    updated_data["totalspend"] = current_customer_data.get("totalspend", 0) + new_transaction_data.get("amount", 0)
    updated_data["totalpurchases"] = current_customer_data.get("totalpurchases", 0) + 1
    
    # Update recency (days since last purchase)
    if "date" in new_transaction_data:
        last_date = pd.to_datetime(new_transaction_data["date"])
        ref_date = datetime.now()
        updated_data["recency_days"] = (ref_date - last_date).days
    
    # Recalculate purchase_rate if tenure is available
    if "tenure_days" in current_customer_data:
        updated_data["purchase_rate"] = updated_data["totalpurchases"] / (current_customer_data["tenure_days"] + 1)
    
    # Recalculate spend_per_purchase
    if updated_data["totalpurchases"] > 0:
        updated_data["spend_per_purchase"] = updated_data["totalspend"] / updated_data["totalpurchases"]
    else:
        updated_data["spend_per_purchase"] = 0
    
    # Re-assign segment
    return assign_segment(updated_data)

# Example: Customer makes a new purchase
print("Example: Customer update after new transaction")
print("-" * 60)

current_customer = {
    "totalspend": 1000,
    "totalpurchases": 10,
    "purchasedays": 5,
    "recency_days": 30,
    "tenure_days": 100,
    "spend_per_purchase": 100
}

new_transaction = {
    "amount": 50,
    "quantity": 1,
    "category": "Electronics",
    "date": "2021-12-15"
}

updated_segment = update_customer_segment("CUST_123", new_transaction, current_customer)
print(f"\nUpdated segment assignment:")
for key, value in updated_segment.items():
    print(f"  {key}: {value}")


## Step 7: Summary of Thresholds and Segment Definitions


In [ ]:
# --- Create summary table of thresholds and segments ---

print("="*80)
print("RULES-BASED SEGMENTATION SUMMARY")
print("="*80)

print("\nTHRESHOLDS FOR SEGMENT ASSIGNMENT (from cluster boundaries):")
print("-" * 80)
print("  These thresholds are derived from actual cluster boundaries (min/max),")
print("  not arbitrary medians. They better separate the clusters.")
print()
# Group thresholds by feature
threshold_features = ["totalspend", "totalpurchases", "recency_days", "spend_per_purchase"]
for feature in threshold_features:
    low_key = f"{feature}_low"
    high_key = f"{feature}_high"
    if low_key in THRESHOLDS and high_key in THRESHOLDS:
        print(f"  {feature}:")
        print(f"    Low threshold: {THRESHOLDS[low_key]:.2f}")
        print(f"    High threshold: {THRESHOLDS[high_key]:.2f}")
        print()

print("\n\nSEGMENT DEFINITIONS (4 Segment Types):")
print("-" * 80)
for cluster_id in sorted(SEGMENT_STRATEGIES.keys()):
    strategy = SEGMENT_STRATEGIES[cluster_id]
    print(f"\nCluster {cluster_id}: {strategy['segment_type']}")
    print(f"  Characteristics:")
    print(f"    - Avg Frequency: {strategy['avg_frequency']:.1f} purchases")
    print(f"    - Avg Spending: ${strategy['avg_spending']:.2f}")
    print(f"    - Avg Recency: {strategy['avg_recency']:.1f} days")
    print(f"    - Avg Spend per Purchase: ${strategy.get('avg_spend_per_purchase', 0):.2f}")
    print(f"  Strategy: {strategy['recommendation_type']}")
    print(f"  Message: {strategy['message']}")
    print(f"  Discount: {strategy['discount']}%")

print("\n\nNEW CUSTOMER HANDLING:")
print("-" * 80)
print("  All new customers are assigned to: Low-Value, Low-Engagement Customers")
print("  Rationale: Assume low frequency, low spending to decrease churn and attract them")
print("  Strategy: Strong price incentives, re-activation campaigns, discounted popular items")
print("  Discount: 20% off first purchase")
print("  Message: Welcome offer to incentivize first purchase")

print("\n\nRE-ASSIGNMENT RULES:")
print("-" * 80)
print("  After each transaction, customer metrics are updated:")
print("    - totalspend += transaction_amount")
print("    - totalpurchases += 1")
print("    - recency_days = days since last purchase")
print("    - purchase_rate = totalpurchases / (tenure_days + 1)")
print("    - spend_per_purchase = totalspend / totalpurchases")
print("  Segment is re-assigned based on updated metrics using RFM thresholds above")


In [ ]:
import pandas as pd
import numpy as np

# --- Business Value & Revenue Uplift Calculation ---

# 1. Setup Validation Data
# We need to link the training clusters (2018-19) to future performance (2020-21)

# Create a copy to avoid modifying original data
val_df = df_train_clean.copy()

# Ensure 'respondentid' is a column
if 'respondentid' not in val_df.columns:
    val_df = val_df.reset_index()
    if 'respondentid' not in val_df.columns and 'sum_18_final' in locals():
        val_df['respondentid'] = sum_18_final.loc[val_df.index, 'respondentid']

# Rename columns for clarity
val_df = val_df[['respondentid', 'cluster', 'totalspend']].rename(columns={'totalspend': 'Spend_18_19'})

# 2. Merge with Future Data (2020-21)
if 'summary_2021' not in locals():
    try:
        summary_2021 = pd.read_csv('./Ecommerce data/summary2020-21.csv')
    except:
        print("Warning: summary_2021 not found in locals or path.")

val_df = val_df.merge(
    summary_2021[['respondentid', 'totalspend']].rename(columns={'totalspend': 'Spend_20_21'}),
    on='respondentid',
    how='left'
)

# Handle Churn (NaN spend in 2020-21 means 0 spend)
val_df['Spend_20_21'] = val_df['Spend_20_21'].fillna(0)

# 3. Calculate Performance Metrics by Cluster
cluster_perf = val_df.groupby('cluster').agg({
    'respondentid': 'count',
    'Spend_18_19': 'mean',
    'Spend_20_21': 'mean'
}).rename(columns={'respondentid': 'Count'})

# Calculate Organic Growth (without intervention)
cluster_perf['Organic_Growth'] = (cluster_perf['Spend_20_21'] - cluster_perf['Spend_18_19']) / cluster_perf['Spend_18_19']

print("="*80)
print("VALIDATION RESULTS: 2020-21 Performance by Cluster")
print("="*80)
print(cluster_perf.round(2).sort_values('Spend_18_19', ascending=False))
print("\n")

# 4. Dynamic Strategy Mapping (Using Standardized Segment Names)
# We apply lift assumptions based on spending rank, but use the STRICT segment names from Step 4.

sorted_clusters = cluster_perf.sort_values('Spend_18_19', ascending=False).index.tolist()
lift_map = {}

print("="*80)
print("STRATEGY & LIFT ASSUMPTIONS (By Standardized Segment)")
print("="*80)

for rank, cluster_id in enumerate(sorted_clusters):
    # Retrieve the official segment name from the SEGMENT_STRATEGIES dictionary
    # Fallback to a generic name if dictionary is missing (defensive coding)
    if 'SEGMENT_STRATEGIES' in locals() and cluster_id in SEGMENT_STRATEGIES:
        segment_name = SEGMENT_STRATEGIES[cluster_id]['segment_type']
    else:
        segment_name = f"Cluster {cluster_id} (Name Unknown)"

    # Define Lift Logic based on Rank (Spend):
    # Rank 0 (Highest Spend) -> Maintenance/Defense (Lower Lift)
    # Ranks 1-3 -> Growth/Activation (Higher Lift)
    if rank == 0:
        lift_map[cluster_id] = 0.01
        strategy_desc = "Retention/Defense"
    else:
        lift_map[cluster_id] = 0.05
        strategy_desc = "Growth/Activation"
    
    print(f"Cluster {cluster_id}: {segment_name}")
    print(f"  -> Strategy: {strategy_desc}")
    print(f"  -> Assumed Lift: {int(lift_map[cluster_id]*100)}%")
    print("-" * 40)

# 5. Calculate Revenue Uplift
impact_df = cluster_perf.copy()

# Add the Segment Name to the dataframe for final display
if 'SEGMENT_STRATEGIES' in locals():
    impact_df['Segment_Type'] = [SEGMENT_STRATEGIES.get(i, {}).get('segment_type', str(i)) for i in impact_df.index]
else:
    impact_df['Segment_Type'] = impact_df.index

impact_df['Assumed_Lift'] = [lift_map[x] for x in impact_df.index]

# Formula: Uplift = Future Revenue Baseline * Lift %
impact_df['Projected_Revenue_Base'] = impact_df['Count'] * impact_df['Spend_20_21']
impact_df['Revenue_Uplift'] = impact_df['Projected_Revenue_Base'] * impact_df['Assumed_Lift']

# 6. Final Output
print("\n" + "="*80)
print("FINAL BUSINESS VALUE ESTIMATION")
print("="*80)
display_cols = ['Segment_Type', 'Count', 'Spend_20_21', 'Assumed_Lift', 'Revenue_Uplift']
pd.set_option('display.max_colwidth', None) # Ensure full segment names are visible

# Sort by Revenue Uplift to show impact
final_view = impact_df[display_cols].round(2).sort_values('Revenue_Uplift', ascending=False)
print(final_view)

total_uplift = impact_df['Revenue_Uplift'].sum()
print(f"\n>>> TOTAL PROJECTED ANNUAL UPLIFT: ${total_uplift:,.0f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. 准备数据
# 数据来源于之前的有机增长率计算结果
data = {
    'Cluster_ID': ['Cluster 1', 'Cluster 2', 'Cluster 3', 'Cluster 0'],
    'Segment Name': [
        'Mid-Tier\nStable Buyers', 
        'High-Frequency,\nLow-Spend', 
        'High-Value,\nHigh-Engagement', 
        'Low-Value,\nLow-Engagement'
    ],
    'Organic Growth': [1.72, 0.53, 0.28, 0.08]
}
df_plot = pd.DataFrame(data)

# 2. 排序
# 按照增长率降序排列，使视觉重心集中在增长最快的群体
df_plot = df_plot.sort_values('Organic Growth', ascending=False)

# 3. 设置画布
# 比例设置为 8x4.5 英寸，适合半页 PPT 或文档展示
plt.figure(figsize=(6, 4.5))
sns.set_theme(style="whitegrid")

# 4. 绘图
# 使用 seaborn 的 barplot
# palette 可以根据您的品牌色调整，这里使用了清晰的 'viridis'
ax = sns.barplot(x='Segment Name', y='Organic Growth', data=df_plot, palette='viridis')

# 5. 标注数值 (172%, 53% 等)
for p in ax.patches:
    # 获取柱子高度
    height = p.get_height()
    # 在柱子上方标注百分比
    ax.annotate(f'{height:.0%}', 
                (p.get_x() + p.get_width() / 2., height), 
                ha='center', va='center', 
                xytext=(0, 8), # 稍微向上偏移
                textcoords='offset points',
                fontweight='bold',
                fontsize=11)

# 6. 标注 Cluster ID
# 将 Cluster ID 和 Segment Name 结合作为 x 轴标签
# 这样不仅看到了业务名称，也明确了它是哪个 Cluster
x_labels = [f"{row['Cluster_ID']}\n{row['Segment Name']}" for i, row in df_plot.iterrows()]
ax.set_xticklabels(x_labels, fontsize=10)

# 7. 调整标题和轴标签
plt.title('Organic Revenue Growth by Customer Segment (2018-2021)', fontsize=14, fontweight='bold', pad=20)
plt.ylabel('Organic Growth Rate', fontsize=11)
plt.xlabel('') # x轴标签已包含在刻度中，这里留空
plt.ylim(0, 2.0) # 设置y轴上限，留出空间给标签

# 8. 优化布局并保存
plt.tight_layout()
plt.savefig('organic_growth_cluster_bar.png', dpi=300)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.ticker as ticker

# 1. 准备数据
# 根据您的指示，High-Frequency High-Spend 对应之前的 High-Value High-Engagement
data = {
    'Cluster_ID': ['Cluster 2', 'Cluster 1', 'Cluster 3', 'Cluster 0'],
    'Segment Name': [
        'High-Frequency,\nLow-Spend',   # 换行符用于优化显示
        'Mid-Tier\nStable Buyers', 
        'High-Value,\nHigh-Engagement', 
        'Low-Value,\nLow-Engagement'
    ],
    'Revenue Uplift': [437801, 319800, 43775, 23185],
    'Contribution': [0.53, 0.39, 0.05, 0.03] # 这里的占比用于标签显示
}

df_plot = pd.DataFrame(data)

# 2. 排序
# 确保贡献最大的排在最上方
df_plot = df_plot.sort_values('Revenue Uplift', ascending=False)

# 3. 设置画布
# 宽长比设置为 10:5，适合 PPT 横向排版
plt.figure(figsize=(8, 5))
sns.set_theme(style="whitegrid")

# 4. 绘图
# 使用横向条形图 (x=数值, y=分类)
# palette='magma_r' 或 'viridis' 能很好地表达数值强度
ax = sns.barplot(x='Revenue Uplift', y='Segment Name', data=df_plot, palette='viridis')

# 5. 精细化标注
for i, p in enumerate(ax.patches):
    width = p.get_width()
    # 获取对应的百分比数据
    pct_val = df_plot.iloc[i]['Contribution']
    
    # 构造标签: "$437,801 (53%)"
    label_text = f"${width:,.0f} ({pct_val:.0%})"
    
    # 在条形图末端添加标签
    ax.annotate(label_text, 
                (width, p.get_y() + p.get_height() / 2.), 
                ha='left', va='center', 
                xytext=(8, 0), # 向右偏移一点距离
                textcoords='offset points',
                fontweight='bold',
                fontsize=12,
                color='black')

# 6. 优化 Y 轴标签
# 将 Cluster ID 加入标签，显得更专业
y_labels_clean = [f"{row['Cluster_ID']}: {row['Segment Name'].replace(chr(10), ' ')}" for i, row in df_plot.iterrows()]
ax.set_yticklabels(y_labels_clean, fontsize=11, fontweight='medium')

# 7. 标题与格式
plt.title('Projected Annual Revenue Uplift by Customer Segment', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Estimated Annual Revenue Increase ($)', fontsize=12)
plt.ylabel('') # 移除 y 轴标题

# X轴金额格式化
ax.xaxis.set_major_formatter(ticker.StrMethodFormatter('${x:,.0f}'))

# 留出右侧空间给标签
plt.xlim(0, 550000) 

# 8. 保存
plt.tight_layout()
plt.savefig('revenue_uplift_bar_chart.png', dpi=300)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. 准备数据
data = {
    'Cluster_ID': ['Cluster 2', 'Cluster 1', 'Cluster 3', 'Cluster 0'],
    'Segment Name': [
        'High-Frequency,\nLow-Spend', 
        'Mid-Tier\nStable Buyers', 
        'High-Value,\nHigh-Engagement', 
        'Low-Value,\nLow-Engagement'
    ],
    'Revenue Uplift': [437801, 319800, 43775, 23185]
}

df_plot = pd.DataFrame(data)
# 按金额降序排列，保证视觉顺时针递减
df_plot = df_plot.sort_values('Revenue Uplift', ascending=False)

# 2. 设置画布与配色
fig, ax = plt.subplots(figsize=(10, 6))
# 使用 viridis 调色板，颜色区分度高且商务
colors = sns.color_palette('viridis', len(df_plot))

# 3. 绘制甜甜圈图 (Donut Chart)
# wedgeprops={'width': 0.4} 控制环的厚度，让它变成环形
wedges, texts, autotexts = ax.pie(
    df_plot['Revenue Uplift'], 
    labels=None,            # 图上不直接显示长标签，避免拥挤
    autopct='%1.1f%%',      # 显示百分比
    startangle=90,          # 从12点钟方向开始
    colors=colors,
    pctdistance=0.8,        # 百分比数值离圆心的距离
    wedgeprops=dict(width=0.4, edgecolor='w', linewidth=2), # 白色描边增加质感
    textprops=dict(color="w", fontsize=11, fontweight='bold')
)

# 4. 优化显示细节
# 如果占比小于 5%，隐藏图上的百分比数字，防止重叠（通过图例查看）
total_revenue = df_plot['Revenue Uplift'].sum()
for i, autotext in enumerate(autotexts):
    if df_plot.iloc[i]['Revenue Uplift'] / total_revenue < 0.05:
        autotext.set_visible(False)

# 5. 在圆心添加总金额 (Center Text)
ax.text(0, 0, f'Total Uplift\n${total_revenue:,.0f}', 
        ha='center', va='center', 
        fontsize=16, fontweight='bold', color='#333333')

# 6. 构造右侧详细图例 (Legend)
# 图例包含：Cluster ID + 名称 + 金额
legend_labels = [
    f"{row['Cluster_ID']}: {row['Segment Name'].replace(chr(10), ' ')} — ${row['Revenue Uplift']:,.0f}"
    for i, row in df_plot.iterrows()
]

ax.legend(wedges, legend_labels,
          title="Segment Contribution",
          title_fontsize=12,
          loc="center left",
          bbox_to_anchor=(1, 0, 0.5, 1), # 将图例移到圆环右侧
          frameon=False, # 去掉图例边框
          fontsize=11)

# 7. 标题与保存
plt.title('Projected Revenue Uplift Distribution', fontsize=16, fontweight='bold', x=0.5, y=1.05)
plt.tight_layout()

# 保存图片
plt.savefig('revenue_uplift_donut_chart.png', dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
print('\'')